In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from data_loader import (
    load_all_splits, verify_data,
    CLASSES, SORTED_CLASSES, DATA_DIR,
)

print('Classes (alphabetical, index = label):')
for i, c in enumerate(SORTED_CLASSES):
    print(f'  {i} = {c}')

Classes (alphabetical, index = label):
  0 = construction_site
  1 = debris_or_rubble
  2 = fire_station
  3 = flooded_road
  4 = hospital
  5 = impoverished_settlement
  6 = multi-unit_residential
  7 = single-unit_residential


In [2]:
verify_data()

Data check: C:\Users\rdlou\class\data402\finalproject\data

processed:
  construction_site            3326
  debris_or_rubble             2098
  fire_station                 1348
  flooded_road                 2073
  hospital                     1019
  impoverished_settlement      1366
  multi-unit_residential       1373
  single-unit_residential      1500
  total                       14103

processed_val:
  construction_site              40
  debris_or_rubble               18
  fire_station                   94
  flooded_road                   40
  hospital                       73
  impoverished_settlement        16
  multi-unit_residential         99
  single-unit_residential       200
  total                         580

processed_test:
  construction_site              40
  debris_or_rubble               19
  fire_station                   95
  flooded_road                   40
  hospital                       74
  impoverished_settlement        17
  multi-unit_residential        

In [3]:
X_train, y_train, X_val, y_val, X_test, y_test = load_all_splits(size=64)

Training set:
  construction_site            3326
  debris_or_rubble             2098
  fire_station                 1348
  flooded_road                 2073
  hospital                     1019
  impoverished_settlement      1366
  multi-unit_residential       1373
  single-unit_residential      1500
Validation set:
  construction_site              40
  debris_or_rubble               18
  fire_station                   94
  flooded_road                   40
  hospital                       73
  impoverished_settlement        16
  multi-unit_residential         99
  single-unit_residential       200
Test set:
  construction_site              40
  debris_or_rubble               19
  fire_station                   95
  flooded_road                   40
  hospital                       74
  impoverished_settlement        17
  multi-unit_residential        100
  single-unit_residential       200

Shapes: train (14103, 12288), val (580, 12288), test (585, 12288)


In [4]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train.astype(np.float32))
X_val_s   = scaler.transform(X_val.astype(np.float32))
X_test_s  = scaler.transform(X_test.astype(np.float32))

print(f'X_train_s: {X_train_s.shape}, dtype={X_train_s.dtype}')

X_train_s: (14103, 12288), dtype=float32


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score

best_f1, best_C, best_clf = -1, None, None

for C in [0.001, 0.01, 0.1, 1.0, 10.0]: # Doing this to see how regularized we should make the model (lower C = more simple / stronger regularization, larger C = opposite)
    clf = LogisticRegression(
        max_iter=1000,
        class_weight='balanced',
        solver='lbfgs',
        C=C,
        n_jobs=-1,
    )
    clf.fit(X_train_s, y_train)
    val_pred = clf.predict(X_val_s)
    acc = (val_pred == y_val).mean()
    f1 = f1_score(y_val, val_pred, average='macro')
    print(f'C={C:<7}  val acc: {acc:.3f}   macro-F1: {f1:.3f}')
    if f1 > best_f1:
        best_f1, best_C, best_clf = f1, C, clf

print(f'\nBest: C={best_C}, val macro-F1={best_f1:.3f}')

| C       | Validation Accuracy | Macro-F1 |
|---------|---------------------|----------|
| 0.001   | 0.234               | 0.203    |
| 0.01    | 0.228               | 0.188    |
| 0.1     | 0.212               | 0.175    |
| 1.0     | 0.184               | 0.153    |
| 10.0    | 0.193               | 0.161    |

**Best:** `C = 0.001`, validation macro-F1 = **0.203**

The sweep selects the strongest regularization in our grid, which makes sense given the model has ~12,000 features (64×64×3 pixels) and only ~14,000 training examples — without aggressive regularization, the linear model overfits to pixel-level noise. The fact that the best C sits at the edge of the grid suggests trying even smaller values (e.g., 0.0001, 0.00001) could squeeze out a small additional gain

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

val_pred = best_clf.predict(X_val_s)

print('Validation classification report:')
print(classification_report(y_val, val_pred,
                            target_names=SORTED_CLASSES, digits=3))

print('\nValidation confusion matrix:')
cm = confusion_matrix(y_val, val_pred)
print(pd.DataFrame(cm, index=SORTED_CLASSES, columns=SORTED_CLASSES))

### Validation Classification Report

| Class                   | Precision | Recall | F1-Score | Support |
|-------------------------|-----------|--------|----------|---------|
| construction_site       | 0.111     | 0.175  | 0.136    | 40      |
| debris_or_rubble        | 0.062     | 0.222  | 0.096    | 18      |
| fire_station            | 0.304     | 0.298  | 0.301    | 94      |
| flooded_road            | 0.234     | 0.450  | 0.308    | 40      |
| hospital                | 0.259     | 0.192  | 0.220    | 73      |
| impoverished_settlement | 0.045     | 0.125  | 0.067    | 16      |
| multi-unit_residential  | 0.216     | 0.192  | 0.203    | 99      |
| single-unit_residential | 0.454     | 0.220  | 0.296    | 200     |
| **accuracy**            |           |        | **0.234**| **580** |
| **macro avg**           | 0.211     | 0.234  | 0.203    | 580     |
| **weighted avg**        | 0.302     | 0.234  | 0.249    | 580     |

### Validation Confusion Matrix

Rows are true class, columns are predicted class. Abbreviations: CS = construction_site, DR = debris_or_rubble, FS = fire_station, FR = flooded_road, HS = hospital, IS = impoverished_settlement, MR = multi-unit_residential, SR = single-unit_residential.

| True \ Pred | CS  | DR  | FS  | FR  | HS  | IS  | MR  | SR  |
|-------------|-----|-----|-----|-----|-----|-----|-----|-----|
| **CS**      | 7   | 7   | 6   | 4   | 2   | 5   | 5   | 4   |
| **DR**      | 1   | 4   | 1   | 2   | 2   | 1   | 3   | 4   |
| **FS**      | 10  | 4   | 28  | 13  | 7   | 4   | 13  | 15  |
| **FR**      | 5   | 6   | 4   | 18  | 2   | 1   | 2   | 2   |
| **HS**      | 10  | 9   | 4   | 11  | 14  | 6   | 12  | 7   |
| **IS**      | 3   | 2   | 2   | 1   | 1   | 2   | 5   | 0   |
| **MR**      | 11  | 9   | 18  | 5   | 10  | 6   | 19  | 21  |
| **SR**      | 16  | 24  | 29  | 23  | 16  | 19  | 29  | 44  |

### Discussion

The logistic regression baseline achieves a validation macro-F1 of 0.203 — meaningfully above the random-guessing floor (~0.125 for 8 balanced classes) but well below useful performance. Several patterns are worth highlighting:

The two best-performing classes are **flooded_road** (F1 = 0.308, recall 45%) and **fire_station** (F1 = 0.301). The flooded_road result is somewhat surprising given the dataset's known label-noise issue, where multiple temporal views of the same location can show the road both dry and flooded — the linear model still picks up enough water-pixel signal to do better than average. fire_station's structure (often standalone buildings with distinctive parking-lot adjacencies) appears to be linearly separable enough to get reasonable signal from raw pixels.

The worst-performing classes are **impoverished_settlement** (F1 = 0.067) and **debris_or_rubble** (F1 = 0.096), both starved by extremely small validation sets (16 and 18 examples respectively) and by the linear model's inability to capture the spatial texture cues that distinguish these classes from residential areas and construction sites.

The confusion matrix reveals the model's core weakness: **single-unit_residential** has 200 validation examples but only 44 are correctly identified — its predictions are scattered fairly evenly across all other classes, suggesting the model has no coherent decision boundary for it in raw-pixel space. The expected confusion between **single-unit and multi-unit residential** (21 SR predicted as MR, 29 MR predicted as SR) is visible, as is the **debris vs construction_site** confusion that motivated keeping construction_site as a confounder class in our sampling.

These results establish the floor we'd expect from a model that lacks any notion of spatial structure. The CNN and pretrained ResNet should close most of this gap by learning local visual features that no linear model can capture from flattened pixel intensities.

## Convolution Neural Network

In [9]:
import torch
import torchvision
from torchvision import transforms
from torchvision.datasets import ImageFolder
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

True
NVIDIA GeForce RTX 3060 Laptop GPU


### Loading images in as tensors and doing basic transformations to help the CNN

In [13]:
transforms_train = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(degrees=180),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

transforms_val = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

train_ds = ImageFolder('data/processed',      transform=transforms_train)
val_ds   = ImageFolder('data/processed_val',  transform=transforms_val)
test_ds  = ImageFolder('data/processed_test', transform=transforms_val)

print(len(train_ds))
print(train_ds.classes)
img, label = train_ds[0]
print(img.shape, img.dtype)
print(label)

14103
['construction_site', 'debris_or_rubble', 'fire_station', 'flooded_road', 'hospital', 'impoverished_settlement', 'multi-unit_residential', 'single-unit_residential']
torch.Size([3, 224, 224]) torch.float32
0


### Batching the data

In [18]:
from torch.utils.data import DataLoader

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True, num_workers=0)
val_loader = DataLoader(val_ds, batch_size=32, shuffle=False, num_workers=0)
test_loader = DataLoader(test_ds, batch_size=32, shuffle=False, num_workers=0)

images, labels = next(iter(train_loader))
print(images.shape)
print(labels.shape)
print(labels[:10])

torch.Size([32, 3, 224, 224])
torch.Size([32])
tensor([2, 0, 0, 0, 1, 5, 3, 5, 6, 0])


### Making the CNN from scratch class and doing quick test to make sure pipeline works

In [19]:
import torch.nn as nn

class FmowCNN(nn.Module):
    def __init__(self, num_classes=8):
        super().__init__()
        
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(32)
        
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(64)
        
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm2d(128)
        
        self.conv4 = nn.Conv2d(128, 256, kernel_size=3, padding=1)
        self.bn4 = nn.BatchNorm2d(256)
        
        self.relu = nn.ReLU()
        self.pool = nn.MaxPool2d(kernel_size=2)
        
        self.global_pool = nn.AdaptiveAvgPool2d(1)
        self.dropout = nn.Dropout(0.5)
        self.fc = nn.Linear(256, 8)
    
    def forward(self, x):
        x = self.pool(self.relu(self.bn1(self.conv1(x))))
        x = self.pool(self.relu(self.bn2(self.conv2(x))))
        x = self.pool(self.relu(self.bn3(self.conv3(x))))
        x = self.pool(self.relu(self.bn4(self.conv4(x))))
        x = self.global_pool(x)
        x = x.view(x.size(0), -1)
        x = self.dropout(x)
        x = self.fc(x)
        return x

model = FmowCNN(num_classes=8)
print(model)
dummy = torch.randn(2, 3, 224, 224)
output = model(dummy)
print(output.shape)

FmowCNN(
  (conv1): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (bn1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (conv2): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (conv3): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (bn3): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (conv4): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (bn4): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU()
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (global_pool): AdaptiveAvgPool2d(output_size=1)
  (dropout): Dropout(p=0.5, inplace=False)
  (fc): Linear(in_features=256, out_features=8, bias=True)
)
torch.Size([2, 8])


### Setting to gpu for speed

In [20]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

Using device: cuda


### Adding class weights

In [21]:
import numpy as np
train_labels = [label for _, label in train_ds.samples]
class_counts = np.bincount(train_labels)
class_weights = len(train_labels) / (len(class_counts) * class_counts)
class_weights = torch.tensor(class_weights, dtype=torch.float32).to(device)
print('Class weights:', class_weights)

Class weights: tensor([0.5300, 0.8403, 1.3078, 0.8504, 1.7300, 1.2905, 1.2840, 1.1753],
       device='cuda:0')


### Making model instance and loss function + optimizer

In [22]:
model = FmowCNN(num_classes=8).to(device)
criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [25]:
import time
NUM_EPOCHS = 30

best_val_f1 = -1

for epoch in range(NUM_EPOCHS):
    model.train()
    train_loss_sum = 0
    train_correct = 0
    train_total = 0
    
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        logits = model(images)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        
        train_loss_sum += loss.item() * images.size(0)
        train_correct += (logits.argmax(1) == labels).sum().item()
        train_total += labels.size(0)
    
    train_loss = train_loss_sum / train_total
    train_acc = train_correct / train_total
    
    model.eval()
    val_loss_sum = 0
    val_correct = 0
    val_total = 0
    all_preds, all_labels = [], []
    
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            logits = model(images)
            loss = criterion(logits, labels)
            
            val_loss_sum += loss.item() *images.size(0)
            preds = logits.argmax(1)
            val_correct += (preds == labels).sum().item()
            val_total += images.size(0)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
        
        val_loss = val_loss_sum / val_total
        val_acc = val_correct / val_total
        
        from sklearn.metrics import f1_score
        val_f1 = f1_score(all_labels, all_preds, average='macro')
        
        print(f'Epoch {epoch+1:2d}/{NUM_EPOCHS} | '
          f'train loss {train_loss:.3f} acc {train_acc:.3f} | '
          f'val loss {val_loss:.3f} acc {val_acc:.3f} f1 {val_f1:.3f}')
        
    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        torch.save(model.state_dict(), 'best_cnn.pth')
        print(f'  saved new best model (val F1 = {val_f1:.3f})')
        
    time.sleep(5)


Epoch  1/30 | train loss 1.520 acc 0.414 | val loss 1.494 acc 0.453 f1 0.378
Epoch  2/30 | train loss 1.501 acc 0.423 | val loss 1.512 acc 0.426 f1 0.346
Epoch  3/30 | train loss 1.487 acc 0.430 | val loss 1.556 acc 0.434 f1 0.414
Epoch  4/30 | train loss 1.459 acc 0.442 | val loss 1.475 acc 0.455 f1 0.462
Epoch  5/30 | train loss 1.441 acc 0.447 | val loss 1.436 acc 0.445 f1 0.405
Epoch  6/30 | train loss 1.435 acc 0.457 | val loss 1.537 acc 0.426 f1 0.416
Epoch  7/30 | train loss 1.420 acc 0.466 | val loss 1.626 acc 0.388 f1 0.379
Epoch  8/30 | train loss 1.412 acc 0.466 | val loss 1.464 acc 0.419 f1 0.414
Epoch  9/30 | train loss 1.396 acc 0.476 | val loss 1.386 acc 0.464 f1 0.421
Epoch 10/30 | train loss 1.378 acc 0.483 | val loss 1.372 acc 0.490 f1 0.445
Epoch 11/30 | train loss 1.371 acc 0.485 | val loss 1.406 acc 0.476 f1 0.436


KeyboardInterrupt: 

In [ ]:
model.load_state_dict(torch.load('best_cnn.pth'))
model.eval()

all_preds, all_labels = [], []
with torch.no_grad():
    for images, labels in val_loader:
        images = images.to(device)
        preds = model(images).argmax(1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.numpy())

from sklearn.metrics import classification_report, confusion_matrix
print(classification_report(all_labels, all_preds, target_names=SORTED_CLASSES, digits=3))